# chrisMain — EyeMovementTrajectoryAlternatingBackground

Workflow notebook for the `eye_movement_alt_bg` protocol. Cells are grouped into four pieces:

- **Setup** (§1–§3): pick a date, build the pipeline.
- **QC + archive** (§14, §16–§18): protocol QC → visual QC → per-cell
  PNG archive (single date or batch). §17b/§17c are the sorting-QC
  tools: static PNGs for batch review and an interactive ipywidgets
  panel for spot checks.
- **Offline store** (§20): pack QC-good cells into a single HDF5
  per date for fast reload.
- **Analyses** (§21–§22): per-date PSTH / movie-repeat / population
  metrics / Victor-Purpura, plus cross-date pooling.

Exploratory cells (single-cell STA/EI, raster/PSTH spot checks, manual
calibration, EI-match diagnostics) live in git history — they are
either subsumed by `ra.analyze_experiment` or one-off inspection. Use
`git show HEAD~1:demos/chrisMain.ipynb` to retrieve them if needed.


In [1]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

## 1. Find all experiments that ran `AlternatingBackground`

`get_datasets_from_protocol_names` does a lowercase substring match against the protocol registry, so `'varmeannoise'` will catch the full Java class name (e.g. `manookinlab.protocols.VarMeanNoise`). After the DB query we filter to only experiments whose sort output is actually present on disk (`ra.ANALYSIS_DIR`).

In [2]:
exp_search = ra.get_datasets_from_protocol_names('AlternatingBackground')

# Filter to experiments that exist on the MEA SSD
available_experiments = os.listdir(ra.ANALYSIS_DIR)
exp_search = exp_search.query('exp_name in @available_experiments').reset_index(drop=True)

print(f'{len(exp_search)} usable datafile(s) found across '
      f'{exp_search.exp_name.nunique()} experiment(s).')
display(exp_search)


Found 1 protocols matching "alternatingbackground":
['edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground']

Found 22 experiments, 31 epoch blocks.

26 usable datafile(s) found across 19 experiment(s).


,exp_name,datafile_name,NDF,chunk_name,protocol_name,is_mea,data_dir,group_label,experiment_id,protocol_id,group_id,block_id,chunk_id
0,20230214C,data018,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230214C/data018,eye movement alt background ndf 3.0,39,41,781,1484,97
1,20230313C,data017,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data017,eye movement alt background ndf 3.0,42,41,878,1609,108
2,20230313C,data019,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data019,Var Mean Drift Grating ndf 3.0,42,41,879,1611,108
3,20230502C,data018,2.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230502C/data018,ndf 2.0 eye movement alt background,52,41,1077,1855,156
4,20230523C,data013,2.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230523C/data013,Eye movement alt background,56,41,1178,1983,176
5,20230525C,data052,0.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230525C/data052,Eye Movement Alternating Background Photopic,58,41,1220,2041,184
6,20230725C,data036,2.0,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230725C/data036,ndf 2 eye movement alt background,65,41,1397,2264,205
7,20231003C,data021,0.5,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231003C/data021,eye movement alt background,71,41,1516,2416,221
8,20231108C,data005,0.5,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231108C/data005,eye movement alt background,74,41,1556,2462,243
9,20231220C,data022,0.5,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231220C/data022,CC eye movement alt background,76,41,1607,2517,252


## 2. Pick a (date, datafile)

`exp_search` (§1) has **one row per (exp_name, datafile_name) pair**, so multiple datafiles of the same protocol on one date are already separated into distinct rows. Just pick an index into that table — no need to write the date and the datafile separately and risk them drifting apart.

In [ ]:
ENTRY_INDEX = 14   # <-- EDIT ME: row index in exp_search above

_entry = exp_search.iloc[ENTRY_INDEX]
exp_name      = _entry['exp_name']
datafile_name = _entry['datafile_name']
print(f'Selected entry {ENTRY_INDEX}: {exp_name} / {datafile_name}')

# Summary of the chosen day, capped at SUMMARY_HEAD_N rows so long experiments
# don't blow up the notebook output. Increase the cap if you need to inspect
# more rows (or call ra.get_exp_summary(exp_name) directly to see all of them).
SUMMARY_HEAD_N = 10
_sum_df = ra.get_exp_summary(exp_name)
print(f'\nexperiment summary: {len(_sum_df)} datafiles total '
      f'(showing first {min(SUMMARY_HEAD_N, len(_sum_df))})')
display(_sum_df.head(SUMMARY_HEAD_N))


## 3. Build the pipeline

`create_mea_pipeline` does everything in one shot:
- **StimBlock** — loads stimulus params and frame timing for the protocol datafile.
- **ResponseBlock** — loads spike times from the kilosort output for that datafile.
- **Noise chunk** — by default auto-resolved to the chunk closest in time (see `stim.py:get_nearest_noise`). **Set `noise_chunk_name = 'chunkN'`** at the top of the cell below to override this with a specific noise chunk — useful when the auto-pick lands on a chunk that ran *after* the protocol, or when you want to use an earlier sort.
- **Typing file** — set `typing_file_name = 'kilosort*.classification.txt'` to pick a specific classification file when the chunk has more than one.
- **AnalysisChunk** — loads STAs, RF params, EIs, ISIs, timecourses, and the classification (cell type) file from the noise chunk.
- **EI-based cluster match** — `cluster_match()` is called internally to align the noise-chunk cell IDs to the protocol cell IDs by EI footprint correlation. After this, the protocol cells carry `noise_id` and `cell_type` columns.

In [ ]:
# Older experiments are often sorted with kilosort2 (not 2.5). Auto-detect:
exp_sort_dir = os.path.join(ra.DATA_DIR, exp_name, datafile_name)
available_ss = [d for d in os.listdir(exp_sort_dir) if d.startswith('kilosort')]
ss_version = 'kilosort2.5' if 'kilosort2.5' in available_ss else available_ss[0]

# ---- USER INPUT --------------------------------------------------------
# Noise chunk to use for cell typing + EI matching.
#   None  → MEAStimBlock.nearest_noise_chunk (closest in time, either direction)
#   'chunkN' → pin explicitly. Bypasses get_nearest_noise.
noise_chunk_name = None
typing_file_name = None           # e.g. 'kilosort2.5.classification.txt', or None for first match

# EI cluster-match knobs (forwarded to vision_utils.cluster_match).
ei_corr_cutoff       = 0.65        # minimum max-correlation to accept a match (0.6 looser, 0.9 stricter)
ei_match_method      = 'all'      # 'all' (max of three), 'full', 'space', 'power'
ei_use_isi           = False      # also require ISI corr ≥ 0.3
ei_use_timecourse    = False      # also require RGB timecourse corr ≥ 0.3
ei_n_removed_channels = 1         # drop this many top-amplitude electrodes per EI before correlating
# ------------------------------------------------------------------------

# DB record: each datafile row carries a chunk_name written at ingest time.
# That's what the experimenter declared at the rig. Use it as a sanity check
# against whatever auto-pick or user-pin we end up using below.
_exp_df = ra.get_exp_summary(exp_name)
_db_chunk_row = _exp_df.query('datafile_name == @datafile_name')
db_chunk_name = None
if not _db_chunk_row.empty:
    db_chunk_name = str(_db_chunk_row['chunk_name'].iloc[0])
    print(f'database chunk_name for {datafile_name}: {db_chunk_name}')

# Resolve noise chunk: explicit override > auto-pick via MEAStimBlock.
# When pinned, MEAStimBlock will skip get_nearest_noise() inside the
# pipeline build below — no need to construct a throwaway block here.
if noise_chunk_name is None:
    from retinanalysis.classes.stim import MEAStimBlock
    _tmp_stim = MEAStimBlock(exp_name, datafile_name, verbose=False)
    noise_chunk_name = _tmp_stim.nearest_noise_chunk
    print(f'auto-picked noise chunk: {noise_chunk_name}')
else:
    print(f'using user-specified noise chunk: {noise_chunk_name}')

# Sanity check: does the chunk we will load match the one the experimenter
# declared in the DB? Mismatch is allowed (the auto-pick or the user might
# correctly want a different chunk) but worth flagging loudly.
if db_chunk_name is not None and db_chunk_name != noise_chunk_name:
    print(f'\n*** ALERT: DB chunk {db_chunk_name!r} differs from chunk being used '
          f'({noise_chunk_name!r}). ***')
    print(f'    If the DB record is correct, set noise_chunk_name = {db_chunk_name!r} '
          f'above and re-run this cell.\n')

# List candidate classification files (skip macOS AppleDouble dotfiles).
chunk_dir = os.path.join(ra.ANALYSIS_DIR, exp_name, noise_chunk_name, ss_version)
typing_candidates = [
    f for f in os.listdir(chunk_dir)
    if f.endswith('.classification.txt') and not f.startswith('.')
]
if not typing_candidates:
    raise FileNotFoundError(
        f'No .classification.txt in {chunk_dir}. '
        f'Pick a different noise_chunk_name above.'
    )
if typing_file_name is None:
    typing_file_name = typing_candidates[0]
elif typing_file_name not in typing_candidates:
    raise FileNotFoundError(
        f'{typing_file_name!r} not found in {chunk_dir}. '
        f'Available: {typing_candidates}'
    )

print(f'ss_version: {ss_version}')
print(f'noise chunk: {noise_chunk_name}')
print(f'typing file: {typing_file_name}')
print(f'(other classification files in chunk: {typing_candidates})')
print(f'EI match: method={ei_match_method!r} cutoff={ei_corr_cutoff} '
      f'use_isi={ei_use_isi} use_timecourse={ei_use_timecourse} '
      f'n_removed_channels={ei_n_removed_channels}')

pipeline = ra.create_mea_pipeline(
    exp_name,
    datafile_name,
    ss_version=ss_version,
    typing_file=typing_file_name,
    analysis_chunk_name=noise_chunk_name,   # honor the override above
    ei_corr_cutoff=ei_corr_cutoff,
    ei_match_method=ei_match_method,
    ei_use_isi=ei_use_isi,
    ei_use_timecourse=ei_use_timecourse,
    ei_n_removed_channels=ei_n_removed_channels,
)

stim_block      = pipeline.stim
response_block  = pipeline.resp
analysis_chunk  = pipeline.analysis_chunk

print(f'\nNoise chunk used: {analysis_chunk.chunk_name}')
print(f'Cells in noise chunk: {len(analysis_chunk.cell_ids)}')
print(f'Cells in protocol datafile: {len(response_block.cell_ids)}')
print(f'Cells matched by EI: {len(pipeline.match_dict)}')
print(f'ei_match_config: {pipeline.ei_match_config}')


database chunk_name for data018: dynamics2
using user-specified noise chunk: dynamics2


FileNotFoundError: No .classification.txt in /Volumes/data/data/sorted/20240523C/dynamics2/kilosort2.5. Pick a different noise_chunk_name above.

## 14. Per-cell QC inside the protocol

A cell can pass classification on the noise chunk and still misbehave inside a downstream protocol — drift off, drop out for runs of trials, or barely fire. `protocol_qc.block_qc_metrics` returns a per-cell metrics DataFrame; `filter_cells_by_qc` adds a boolean `passes` column with sensible defaults (override any threshold per-call).

Two automated gates do most of the work:

- **Adaptive firing rate.** Set `min_rate_hz` (default **1 Hz**); the per-epoch count threshold is `min_rate_hz × epoch_duration_s`. A cell passes the rate gate when at least `min_frac_epochs_above_rate` (default **80%**) of its epochs hit that threshold. Scales with epoch length so the same defaults work across 5 s, 30 s, and 60 s protocols.
- **Silent-epoch survival.** A cell passes when at least `min_frac_non_silent_epochs` (default **2/3**) of its epochs have ≥1 spike. Equivalent to "drop the silent epochs and keep the cell if at least two-thirds of its trials are still present" — without actually removing epochs (per-condition PSTH plumbing downstream stays intact).

Other reportable metrics (gateable by setting their thresholds): `mean_rate_hz`, `min_count_per_epoch`, `cv_count`, `fano` (off by default — scales with mean), `silent_trial_frac`, `silent_run_max` (consecutive zero-spike trials), `drift_score` (across-trial trend), `reliability_r` (split-half PSTH; off by default for mixed-condition protocols).

The QC outcome is persisted to `<OUTPUT_DIR>/<exp>/<protocol>/qc.csv` — the **initial good/bad tagging** for every cell. §16 (visual QC) and §17/§18 (archives) both honor it; the visual layer is purely additive.

In [ ]:
import pandas as pd
from retinanalysis.utils.protocol_qc import (
    block_qc_metrics, filter_cells_by_qc, QCThresholds,
    protocol_qc_csv_path,
)

# ---- USER INPUT --------------------------------------------------------
# When a qc.csv from a prior run already exists for this date, default to
# LOADING it instead of recomputing. Flip OVERWRITE_QC=True to recompute
# even when one exists (e.g. after tweaking the thresholds below).
OVERWRITE_QC = False
# ------------------------------------------------------------------------

# Two adaptive gates — both are user-tunable here.
MIN_RATE_HZ = 1.0                  # firing-rate floor in spikes/s
MIN_FRAC_EPOCHS = 0.8              # fraction of epochs that must meet that rate
MIN_FRAC_NON_SILENT = 2.0 / 3.0    # cell kept iff ≥ this fraction of epochs has ≥1 spike

# Resolve the same protocol subdir §17 will use, so loading matches saving.
from retinanalysis.utils.cell_plot_archive import protocol_short_name
_short = protocol_short_name(response_block.protocol_name)
if 'protocol_subdir' in dir() and protocol_subdir is not None:
    _short = protocol_subdir
elif 'append_datafile_to_subdir' in dir() and append_datafile_to_subdir:
    _short = f'{_short}_{datafile_name}'
_qc_path = protocol_qc_csv_path(exp_name, _short)

if _qc_path.exists() and not OVERWRITE_QC:
    qc = pd.read_csv(_qc_path)
    print(f'Loaded qc.csv from {_qc_path}  ({len(qc)} cells)')
    print('Set OVERWRITE_QC=True above to recompute with current thresholds.')
else:
    if _qc_path.exists():
        print(f'OVERWRITE_QC=True: recomputing and overwriting {_qc_path}')
    else:
        print(f'No qc.csv on disk yet — computing fresh.')

    t_total_ms = (
        response_block.d_timing['pre_time_ms']
        + response_block.d_timing['stim_time_ms']
        + response_block.d_timing['tail_time_ms']
    )
    qc_metrics = block_qc_metrics(
        response_block, t_start_ms=0, t_end_ms=t_total_ms,
        min_rate_hz=MIN_RATE_HZ,
    )
    qc = filter_cells_by_qc(qc_metrics, thresholds=QCThresholds(
        min_rate_hz=MIN_RATE_HZ,
        min_frac_epochs_above_rate=MIN_FRAC_EPOCHS,
        min_frac_non_silent_epochs=MIN_FRAC_NON_SILENT,
    ))
    # Persist this date's QC outcome — initial good/bad tag set for every cell.
    qc_path = ra.save_protocol_qc(qc, exp_name, protocol=_short)
    print(f'qc.csv → {qc_path}')

epoch_s = qc['epoch_duration_s'].iloc[0]
print(f'\nepoch window: {epoch_s:.1f} s')
print(f'  rate gate:        ≥ {MIN_RATE_HZ:.1f} Hz × {epoch_s:.1f} s = '
      f'{MIN_RATE_HZ*epoch_s:.0f} spikes/epoch in ≥{100*MIN_FRAC_EPOCHS:.0f}% of epochs')
print(f'  silent-epoch gate: ≥ {100*MIN_FRAC_NON_SILENT:.0f}% of epochs have ≥1 spike')

n_pass = int(qc.passes.sum())
print(f'\nTotal cells: {len(qc)},  Passing QC: {n_pass} ({100*n_pass/len(qc):.1f}%)')
print('\nPass rate by cell type:')
for ct, sub in qc.groupby('cell_type'):
    rate = sub['mean_rate_hz'].median()
    print(f'  {ct:<12}  {sub.passes.sum():>4} / {len(sub):>4}  '
          f'({100*sub.passes.mean():3.0f}%)   median rate: {rate:5.1f} Hz')

# Show the most informative failures: those that survive the rate gate
# but fail the silent-epoch gate (or vice versa) tell you which gate did
# the work for a given cell.
fails = qc[~qc.passes].sort_values('frac_non_silent_epochs')
print(f'\nFirst few failing cells ({len(fails)} total):')
display(fails[['cell_id', 'cell_type', 'n_epochs', 'mean_rate_hz',
               'frac_epochs_above_rate', 'frac_non_silent_epochs',
               'silent_run_max', 'drift_score']].head().round(2))


## 16. Visual QC (optional) — click through each cell, tag good/bad

**This step is optional.** §14 wrote an automated QC pass/fail to `qc.csv`. Use the visual-QC GUI when you want to **further restrict** the archive by eyeballing each cell.

The workflow is intentionally iterative:

1. First time through, skip this section (no PNGs exist yet) and run §17 / §18 to build the initial archive.
2. Come back here once PNGs are on disk — `ra.browse_cells_qc(exp_name)` opens a widget that pages through cells (raster left, PSTH right) with `Good` / `Bad` / `Prev` / `Next` buttons. Each click writes a row to `<OUTPUT_DIR>/<exp>/<protocol>/visual_qc.csv`; the session is resumable.
3. Re-run §17 / §18. They auto-detect `visual_qc.csv` and restrict the per-cell archive to cells tagged `good`. `cell_match.csv` is left comprehensive so downstream EI-stats joins still see the full population.

Downstream selection in population analyses (no change):

```python
cells = ra.select_good_cells()   # uses visual_qc.csv if present, else QC-pass set
```

Requirements: `ipywidgets` (already in the `retinanalysis` kernel).

In [25]:
# Launch the per-cell GUI for the date picked in §2. If no PNGs exist
# yet, the widget prints a message and returns — run §17 (single date) or
# §18 (batch) first to build the archive, then come back here to tag.
ra.browse_cells_qc(exp_name)

## 17. Archive the picked date (single date)

§15 above only renders a handful of demo cells. To write the **full** per-cell PNG archive for the date selected in §2 — `mosaic.png`, `index.csv`, `cell_match.csv`, and `cells/<celltype>/cell_<id>_{raster,psth}.png` — call `ra.analyze_experiment(exp_name, datafile_name)`. This is the same driver §18 uses internally per date in its batch loop; running it here builds the archive for *just* the one date you've been inspecting, before committing to the full batch.

**Visual-QC integration is automatic.** If `visual_qc.csv` exists for this experiment (from §16), the call below restricts the archive to **cells labeled `good`** — no extra flags needed. Otherwise it falls back to every cell that passed §14's automated QC (first-pass default). `cell_match.csv` is left comprehensive either way; visual QC is a downstream filter, not data loss.

In [ ]:
# Full archive for the date picked in §2. analyze_experiment now reads
# visual_qc.csv on its own (respect_visual_qc=True by default) and
# restricts the per-cell PNG step to cells tagged 'good' when the file
# exists — no extra notebook glue needed. overwrite=True regenerates
# every targeted PNG.

# ---- USER INPUT --------------------------------------------------------
# Subdirectory under <OUTPUT_DIR>/<exp_name>/. Default uses the protocol
# short name (e.g. 'eye_movement_alt_bg') — fine when the date has one
# datafile of this protocol. When multiple datafiles of the SAME protocol
# exist on this date, set protocol_subdir below (or set
# append_datafile_to_subdir=True) to disambiguate, otherwise each run
# overwrites the previous one.
protocol_subdir = None             # e.g. 'eye_movement_alt_bg_data032', or None
append_datafile_to_subdir = False  # True → auto-append datafile_name
# ------------------------------------------------------------------------

result = ra.analyze_experiment(
    exp_name,
    datafile_name=datafile_name,
    overwrite=True,
    fit_calibration=False,
    n_jobs=-1,
    verbose=True,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
)
print(f'\nDone: {result["exp_name"]} / {result["datafile_name"]} — '
      f'QC-pass pool: {result["n_cells_passed_qc"]}/{result["n_cells_total"]}')
print(f'  output_dir: {result["output_dir"]}')


## 17b. Spike-sorting QC — raw traces with multi-cell asterisks

PSTH/raster plots tell you whether spike *times* are consistent with the stimulus; they don't tell you whether **the spikes were assigned to the right cell** in the first place. This cell pulls a few sample cells (top firing rate, tagged `good` in §16) and plots the raw voltage on their primary electrode for a few epochs, with:

- ⭐ **red asterisks at the top** marking spikes assigned to the *target* cell,
- ⭐ **other-color asterisks** for every other cell whose EI peaks on the same electrode — so you can see if a neighboring unit is potentially stealing spikes.

Loading raw `.bin` data is slow (a 30-s epoch on a 512-channel array is ~1 GB), so we cap to `N_CELLS × N_EPOCHS` panels and reuse each loaded epoch across all cells. Increase the caps if you want a denser sample.

In [ ]:
# §17b — Sorting QC via raw traces, saved to disk as high-DPI PNGs.
# Samples from QC-pass ∩ visual-QC 'good' cells per type and writes one PNG
# per cell to <OUTPUT>/<exp>/<protocol_subdir>/sorting_qc_<protocol>_<datafile>/.
# Each PNG has N_EPOCHS full-width rows; every row = thin raster strip +
# 300 Hz HP-filtered trace with red marks at the cell's spike times.

CELL_TYPES         = ['OnP', 'OnM']    # which types to sample
N_CELLS_PER_TYPE   = 3                  # cells per type
N_EPOCHS           = 4                  # full epochs to show per cell
SAMPLE_STRATEGY    = 'random'           # 'random' (default) or 'top_rate'
RANDOM_SEED        = None               # int for reproducible sampling; None = fresh
DPI                = 250                # 200–300 is good for visual inspection
OVERWRITE_QC_PNGS  = True               # re-render existing PNGs

sample_df, png_paths = ra.sample_and_plot_sorting_qc(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
    cell_types=CELL_TYPES,
    n_cells_per_type=N_CELLS_PER_TYPE,
    n_epochs=N_EPOCHS,
    sample_strategy=SAMPLE_STRATEGY,
    random_seed=RANDOM_SEED,
    dpi=DPI,
    overwrite=OVERWRITE_QC_PNGS,
)
print(f'\n→ wrote {len(png_paths)} PNG(s); open them with the system viewer.')


## 17c. Interactive sorting QC GUI (ipywidgets)

Reads only a sub-window of one epoch at a time — designed for remote-NAS sessions where loading a full epoch (~1 GB) is wasteful. The bandwidth chip (green = local SSD, amber = network mount) ticks up only when real bytes are read; switching electrode rank, toggling spike overlay, or tweaking appearance (line width, HP cutoff, y-range, vector vs raster output) re-renders from cache with zero I/O.


In [ ]:
from IPython.display import display

# Launches an ipywidgets panel for the pipeline built in §3.
# Pick cell → epoch → top-3 electrode → window (slider or
# FloatText) → 'Load raw trace'. Appearance accordion exposes
# trace/marker style and HP-cutoff frequency. The bandwidth
# meter at the bottom tracks cumulative MB read from disk.
display(ra.sorting_qc_gui(response_block))


## 18. Run the archive for one or many dates (standalone)

This section is **self-contained** — run cell 1 (imports), then jump straight here. `ra.analyze_experiments` packages every step the earlier cells did manually into a single call per date: ss_version + typing file + datafile auto-detect, pipeline build, optional rig calibration, type normalization, QC, composite `mosaic.png` (with temporal-filter + ISI rows), and per-cell `cell_<id>_raster.png` + `cell_<id>_psth.png`.

**Visual-QC integration:** for any date that already has a `visual_qc.csv` in its archive folder, the batch driver restricts that date's PNG step to cells tagged `good` (same logic as §17). Pass `respect_visual_qc=False` to override.

Runs are parallel (`n_jobs=-1` uses every CPU core). Use `on_error='log'` so a bad date is recorded in the summary instead of aborting the batch.

In [ ]:
# Section 18 is SELF-CONTAINED — you only need cell 1 (imports) to run it.
# It will query the protocol registry on its own and dispatch the archive
# pipeline over every experiment found, in parallel.

import os
import pandas as pd

# ---- USER INPUT --------------------------------------------------------
# Yes/no: should we (re)save figures for every cell?
#   True  → render and OVERWRITE all PNGs (use this to refresh stale plots).
#           Per-date visual_qc.csv (if present) restricts the per-cell
#           PNG step to cells tagged 'good'.
#   False → skip the archive step entirely (just list the batch and stop).
SAVE_FIGURES = True
# ------------------------------------------------------------------------

PROTOCOL_SEARCH = 'AlternatingBackground'   # substring matched against protocol names

# Build the date list from the protocol registry, keeping only experiments
# whose sort output is present on disk.
_exp_search = ra.get_datasets_from_protocol_names(PROTOCOL_SEARCH)
_available = set(os.listdir(ra.ANALYSIS_DIR))
_exp_search = _exp_search[_exp_search['exp_name'].isin(_available)]
batch_dates = _exp_search['exp_name'].unique().tolist()

# Subset variants — uncomment / adapt as needed:
# batch_dates = ['20220823C', '20221123C', '20230502C']                                            # hand-pick
# batch_dates = _exp_search.query("exp_name >= '20230101C'")['exp_name'].unique().tolist()        # by date
# batch_dates = _exp_search.query('NDF == 2.0')['exp_name'].unique().tolist()                     # by NDF

print(f'Batch run over {len(batch_dates)} dates: {batch_dates}')
print(f'SAVE_FIGURES = {SAVE_FIGURES}  '
      f'({"overwrite all PNGs" if SAVE_FIGURES else "skip archive step"})')

if not SAVE_FIGURES:
    print('SAVE_FIGURES is False — not calling ra.analyze_experiments. '
          'Set SAVE_FIGURES = True above to (re)render PNGs.')
else:
    results = ra.analyze_experiments(
        batch_dates,
        protocol_search=PROTOCOL_SEARCH,    # resolves datafile per date
        fit_calibration=False,              # True on first pass to seed calibrations
        overwrite=True,                     # resave every PNG (driven by SAVE_FIGURES)
        n_jobs=-1,                          # all CPU cores for per-cell rendering
        on_error='log',                     # keep going past per-date failures
        respect_visual_qc=True,             # restrict to good-tagged cells when present
        verbose=True,
    )

    summary = pd.DataFrame([
        {k: r.get(k) for k in
         ['exp_name', 'datafile_name', 'chunk_name',
          'n_cells_total', 'n_cells_passed_qc', 'ndf', 'error']}
        for r in results
    ])
    display(summary)


## 20. Offline data store (build once, reload fast)

After §16/§17 leaves a curated visual-QC set per experiment, `ra.load_or_build_offline` packages everything an analysis needs — metadata, condition table, per-cell spike times, smoothed PSTHs, STA fit, EI summary — into a single HDF5 at `{OUTPUT_DIR}/{exp}/{protocol}/offline.h5`. Subsequent sessions just **load** the file and never touch DataJoint or the SSD pipeline.

- **First call**: builds the pipeline, runs QC, intersects with `visual_qc.csv` (good cells only), writes `offline.h5`. ~1–2 min/date.
- **Re-runs**: reload is ~0.2 s. Pass `overwrite=True` to rebuild.

The returned `OfflineDataset` exposes:

| attribute / method | what it is |
|---|---|
| `ds.meta`, `ds.timing` | scalars (experiment id, datafile, ndf, preTime/stimTime/sample_rate) |
| `ds.epochs` | DataFrame, one row per epoch + condition columns |
| `ds.cells` | DataFrame, one row per saved cell (cell_type, STA fit, EI stats) |
| `ds.spike_times(cell_id)` | list of arrays (ms), one per epoch |
| `ds.psth_matrix(cell_id)` | `(n_epochs, n_bins)` Hz (Gaussian-smoothed) |
| `ds.psth_time_ms()` | shared bin-center time axis |


In [ ]:
# §20 — Build / load the offline store for one experiment.
# First call: 1–2 min; subsequent calls: <1 s.

EXP = '20221123C'
PROTOCOL = 'eye_movement_alt_bg'

ds = ra.load_or_build_offline(
    EXP, protocol=PROTOCOL,
    protocol_search='AlternatingBackground',
    overwrite=False, verbose=True,
)
print(ds)
print('cell types:', ds.cell_types())
display(ds.epochs.head())
display(ds.cells.head())


## 21. Offline analyses — PSTH avg, spike distance, movie repeat, population metrics

Each analysis takes the `OfflineDataset` and returns a DataFrame (or dict for population metrics). Results are saved next to `offline.h5` as CSV so cross-date aggregation is just `pd.concat`.

- **`analyze_offline`** — per-(cell-type × condition) mean PSTHs (offline equivalent of §15's `eye_movement_alt_bg.analyze`).
- **`spike_distance_analysis`** — Victor-Purpura distance (port of `spkd_with_scr.m`) over a 5-s window per trial. Reports within-condition mean (variability inside a condition) and across-condition mean; **`d_diff = d_across - d_within_avg > 0`** means the condition modulates the response.
- **`movie_repeat_analysis`** — split `stimTime` into cycle-1 vs cycle-2 (15 s + 15 s), drop first second of each, compare per cell × condition: correlation, RMSE, mean-rate ratio (adaptation index), and optional per-trial VP distance.
- **`population_time_scale_metrics`** — time-resolved population vector divergence (Cohen's d, Euclidean / cosine distance, cumulative |Δrate|, per-bin Mann-Whitney AUC) comparing the two `currentBackgroundScale` levels per cell type.

Run `run_protocol_analyses` to compute + save VP + movie-repeat CSVs in one call.


In [ ]:
# §21a — Average PSTH by (cell type × condition). Offline = no DJ needed.
from retinanalysis.protocols import eye_movement_alt_bg as ema

r = ema.analyze_offline(ds, minimum_n=3)
print(f'cell types: {r["cell_types"]}')
print(f'{len(r["conditions"])} conditions, {len(r["time_ms"])} time bins')

ema.plot_psth_by_condition(r, show_individual_cells=False)


In [ ]:
# §21b — Movie-repeat comparison: cycle 1 vs cycle 2 (15s each, drop first 1s).
# compute_vp=False keeps it fast (~30 s); enable for per-trial VP timing differences.
mr = ema.movie_repeat_analysis(
    ds, cycle_sec=15.0, drop_first_sec=1.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    compute_vp=False,
)
print(f'rows: {len(mr)}')
display(mr.groupby('cell_type')[['n_trials',
                                  'rate_cycle1_hz', 'rate_cycle2_hz',
                                  'rate_ratio',
                                  'corr_cycle12', 'rmse_cycle12_hz']]
          .median().round(3))

# Save to disk for cross-date pooling
ema.save_movie_repeat(mr, ds.exp_name)


In [ ]:
# §21c — Population time-scale metrics: two backgroundScale levels per cell type.
# Returns dict {cell_type: {cohens_d_mean, pop_euclid_dist, pop_cosine_dist, ...}}.
import matplotlib.pyplot as plt

pm = ema.population_time_scale_metrics(
    ds, primary_key='currentBackgroundScale',
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    smooth_ms=100.0, minimum_n=3,
)
t_ms = pm['time_ms']
pre = ds.timing['preTime_ms']; stim = ds.timing['stimTime_ms']

types = pm['cell_types']
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ct in types:
    d = pm[ct]
    axes[0].plot(t_ms, d['cohens_d_abs_mean'], label=f'{ct} (n={d["n_cells"]})')
    axes[1].plot(t_ms, d['pop_euclid_dist'], label=ct)
    axes[2].plot(t_ms, d['cum_abs_divergence'], label=ct)
for ax in axes:
    ax.axvline(pre, color='r', lw=0.5, ls='--', alpha=0.5)
    ax.axvline(pre+stim, color='r', lw=0.5, ls='--', alpha=0.5)
axes[0].set_ylabel('mean |Cohen\'s d|')
axes[1].set_ylabel('pop Euclid dist')
axes[2].set_ylabel('cum |Δrate|·dt (spikes·cells)')
axes[2].set_xlabel('time (ms)')
axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle(f'{ds.exp_name}: low vs high background scale, time-resolved')
fig.tight_layout()


In [ ]:
# §21d — Spike-distance (Victor-Purpura) across backgroundScale, within image.
# Default pair_within=('currentImageName',): for each image, pair trials across
# the low vs high backgroundScale.  C-accelerated DP makes this fast — under 5s
# for ~170 cells × 5 images.
sd = ema.spike_distance_analysis(
    ds, window_sec=5.0, cost_per_sec=4.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    pair_within=('currentImageName',),   # hold image constant, compare BG scale
    n_trials_cap=None,                    # use every trial — fast in C
)
print(f'rows: {len(sd)}; cells: {sd["cell_id"].nunique()}; '
      f'images: {sd["group_key"].nunique()}')

display(sd.groupby('cell_type')[['d_within_avg', 'd_across', 'd_diff']]
          .agg(['median', 'count']).round(2))

ema.save_spike_distance(sd, ds.exp_name)


## 22. Cross-date aggregation

Once every experiment has been through §20–§21 (each writes `offline.h5`, `spike_distance.csv`, `movie_repeat.csv` to its own folder), pooling across dates is just a `concat`.

- `ra.load_offline_many()` — dict `{exp_name: OfflineDataset}` for every experiment with `offline.h5` on disk.
- `ema.aggregate_psth_across_dates(offlines)` — pool per-cell mean PSTHs into one (n_cells_total, n_bins) matrix per (cell_type, condition); pass to `ema.plot_psth_by_condition`.
- `ema.load_spike_distance_many()` / `ema.load_movie_repeat_many()` — concatenated long-format DataFrames with `exp_name` tags.


In [ ]:
# §22 — Cross-date pooled analyses.
offlines = ra.load_offline_many()  # all dates with offline.h5
print(f'experiments loaded: {len(offlines)}')
for exp, ds_ in offlines.items():
    print(f'  {exp}: {len(ds_.cell_ids)} cells, types={ds_.cell_types()}')

# Pool PSTHs across dates
pooled = ema.aggregate_psth_across_dates(offlines, minimum_n=5)
print(f'\npooled types: {pooled["cell_types"]} from {pooled["n_dates"]} dates')
ema.plot_psth_by_condition(pooled, show_individual_cells=False)

# Pool spike-distance & movie-repeat CSVs
sd_all = ema.load_spike_distance_many()
mr_all = ema.load_movie_repeat_many()
print(f'\nspike_distance rows: {len(sd_all)} from '
      f'{sd_all["exp_name"].nunique() if not sd_all.empty else 0} dates')
print(f'movie_repeat rows: {len(mr_all)} from '
      f'{mr_all["exp_name"].nunique() if not mr_all.empty else 0} dates')

# Headline cross-date summaries
if not sd_all.empty:
    display(sd_all.groupby(['cell_type'])
                  [['d_within_avg', 'd_across', 'd_diff']]
                  .agg(['median', 'count']).round(2))
if not mr_all.empty:
    display(mr_all.groupby(['cell_type'])
                  [['rate_ratio', 'corr_cycle12', 'rmse_cycle12_hz']]
                  .agg(['median', 'count']).round(3))
